In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re

This file serves as testing environment for evaluating the quality indicators on the reconstructed scores. The file also consists an additional analysis of the medication "brilique", as asked for by the medical expert. For the notebook containing all the analysis and comparisons, refer to ``code\Chadsvasc score extraction\jessa_QI.ipynb``

#  QI1: Proportion AF patients with ChadsVasc calculated
Not possible to answer this QI based on reconstruction of the score. 

# QI2: Proportie antico met positieve Chadsvasc (>=2)


In [ ]:
medications = [
    "lixiana",
    "pradaxa",
    "eliquis",
    "xarelto",
    "marevan",
    "marcoumar",
    "sintrom",
]


def check_medications(sections):
    text = sections.get("ONTSLAG_MEDICATIE", "")
    if text is None or text == "":
        print("Fallback to MEDICATIE section")
        text = sections.get("MEDICATIE", "")
        if text is None:
            return False

    text = text.lower()
    for med in medications:
        if re.search(r"\b" + re.escape(med) + r"\b", text, flags=re.IGNORECASE):
            return True
    return False

In [ ]:
def classify_anticoagulant_use(sections):
    # 1. Retrieve text
    full_text = sections.get("ONTSLAG_MEDICATIE") or sections.get("MEDICATIE") or ""
    if not full_text.strip():
        return "none"

    full_text = full_text.lower()

    # 2. Identify the boundary for stopped medication
    # We look for common headers that indicate the start of the "stopped" list
    stop_headers = [
        "gestopte medicatie",
    ]

    # Find the earliest occurrence of any stop header
    split_index = len(full_text)
    for header in stop_headers:
        idx = full_text.find(header)
        if idx != -1 and idx < split_index:
            split_index = idx

    # Only search in the text BEFORE the stop header
    active_text = full_text[:split_index]

    # 3. Define keywords for temporary duration
    duration_keywords = (
        r"gedurende|voor|tot|ged\.|t\.e\.m\.?|tem\b|t/m\b|staal werd meegegeven"
    )

    found_any = False
    for med in medications:
        # Check for temporary markers within the active line
        # Logic: med name -> search until newline/dash -> check for duration keyword
        temp_pattern = rf"\b{re.escape(med)}\b[^\n-]*?\b({duration_keywords})\b"

        if re.search(temp_pattern, active_text):
            return "temporary"

        # If not temporary, check if it's present at all in the active section
        if re.search(rf"\b{re.escape(med)}\b", active_text):
            found_any = True

    return "permanent" if found_any else "none"

In [ ]:
df = pd.read_feather(r"results/intersection_model.feather")
df_with_score_greater_than_2 = df[
    (df["reconstructed_score_intersection"].notna())
    & (df["reconstructed_score_intersection"] >= 2)
]

df["anticoagulant_status"] = df["sections"].apply(classify_anticoagulant_use)

In [ ]:
# Apply the classification
df_with_score_greater_than_2["anticoagulant_status"] = df_with_score_greater_than_2[
    "sections"
].apply(classify_anticoagulant_use)

# View counts for each type
print(df_with_score_greater_than_2["anticoagulant_status"].value_counts())

# Calculate percentages
total = len(df_with_score_greater_than_2)
permanent_pct = (
    len(
        df_with_score_greater_than_2[
            df_with_score_greater_than_2["anticoagulant_status"] == "permanent"
        ]
    )
    / total
    * 100
)
temporary_pct = (
    len(
        df_with_score_greater_than_2[
            df_with_score_greater_than_2["anticoagulant_status"] == "temporary"
        ]
    )
    / total
    * 100
)

print(f"\nPermanent/Chronic: {permanent_pct:.2f}%")
print(f"Temporary/Durational: {temporary_pct:.2f}%")

anticoagulant_status
permanent    3286
none          820
temporary      72
Name: count, dtype: int64

Permanent/Chronic: 78.65%
Temporary/Durational: 1.72%


# QI3: Proportie antico met negatieve Chadsvasc (<2)


In [ ]:
df_with_score_less_equal_than_2 = df[
    (df["reconstructed_score_intersection"].notna())
    & (df["reconstructed_score_intersection"] < 2)
]
print(len(df_with_score_less_equal_than_2))

# Apply the classification
df_with_score_less_equal_than_2["anticoagulant_status"] = (
    df_with_score_less_equal_than_2["sections"].apply(classify_anticoagulant_use)
)

# View counts for each type
print(df_with_score_less_equal_than_2["anticoagulant_status"].value_counts())

# Calculate percentages
total = len(df_with_score_less_equal_than_2)
permanent_pct = (
    len(
        df_with_score_less_equal_than_2[
            df_with_score_less_equal_than_2["anticoagulant_status"] == "permanent"
        ]
    )
    / total
    * 100
)
temporary_pct = (
    len(
        df_with_score_less_equal_than_2[
            df_with_score_less_equal_than_2["anticoagulant_status"] == "temporary"
        ]
    )
    / total
    * 100
)

print(f"\nPermanent/Chronic: {permanent_pct:.2f}%")
print(f"Temporary/Durational: {temporary_pct:.2f}%")

440
anticoagulant_status
permanent    273
none         117
temporary     50
Name: count, dtype: int64

Permanent/Chronic: 62.05%
Temporary/Durational: 11.36%


In [ ]:
# Filter for permanent cases to verify we aren't missing long-form durations
permanent_samples = df_with_score_less_equal_than_2[
    df_with_score_less_equal_than_2["anticoagulant_status"] == "permanent"
]

print(f"--- Inspecting {min(15, len(permanent_samples))} Permanent Cases ---\n")

for i, row in permanent_samples.head(15).iterrows():
    text = (
        row["sections"].get("ONTSLAG_MEDICATIE")
        or row["sections"].get("MEDICATIE")
        or ""
    )
    print(f"Index {i}:")
    # We print the raw text to see the \n and - markers clearly
    print(text.strip())
    print("-" * 50)